# 14 — Triangle inequality spot-check

Compares several **pairwise distances** on `[unit directions | log1p(Inf.Hyp.Rad)]` (and direction-only variants): **metrics** should give violation rate **~0** (float tolerance); **non-metrics** (cosine “distance”, squared Euclidean) can show **nonzero** violation rates—a diagnostic only, not a bug.


### Triangle inequality check (several distances)

**Method:** One draw of **`trials`** random triples `(a,b,c)`. For each distance `d`, count violations of `d(a,b)+d(b,c)≥d(a,c)` and permutations (`tol=1e-6`). The **same triples** are reused for every distance so rates are comparable.

**Distances:** **`euclidean_X`** — L2 on full `x = [U | log1p(r)]` (metric). **`sqeuclidean_X`** — squared L2 on `x` (non-metric). **`cosine_X`** — `1 − cos∠(x_a,x_b)` on full `x` (non-metric). **`cosine_U`** — same on **unit direction** `U` only (non-metric). **`angular_U`** — `arccos(clip(U_a·U_b,−1,1))`, geodesic on `S³` (metric).

**How to read the output:** **~0** violations on `euclidean_X` and `angular_U` mostly validates code + float noise. **Positive** rates on `sqeuclidean_X` / `cosine_*` show how often those choices break the triangle inequality.


In [ ]:
from pathlib import Path
from typing import Callable

import numpy as np

import dmercator3d_io as dm

merged = dm.load_merged_parquet(Path("cache/merged.parquet"))
rng = np.random.default_rng(11)
U = dm.normalize_direction_nd(merged)
X = np.hstack([U, np.log1p(merged["Inf.Hyp.Rad"]).to_numpy()[:, None]])
n = X.shape[0]
trials = 50_000
tol = 1e-6
triples = rng.integers(0, n, size=(trials, 3))


def _cosine_one_minus_sim(vecs: np.ndarray, i: int, j: int) -> float:
    a = vecs[i]
    b = vecs[j]
    na = float(np.linalg.norm(a))
    nb = float(np.linalg.norm(b))
    if na < 1e-15 or nb < 1e-15:
        return 0.0
    c = float(np.dot(a, b) / (na * nb))
    return float(1.0 - np.clip(c, -1.0, 1.0))


def make_distances() -> dict[str, tuple[Callable[[int, int], float], bool]]:
    """name -> (d(i,j), is_metric_theoretically)"""

    def eucl_X(i: int, j: int) -> float:
        return float(np.linalg.norm(X[i] - X[j]))

    def sqeucl_X(i: int, j: int) -> float:
        dv = X[i] - X[j]
        return float(np.dot(dv, dv))

    def cos_X(i: int, j: int) -> float:
        return _cosine_one_minus_sim(X, i, j)

    def cos_U(i: int, j: int) -> float:
        return _cosine_one_minus_sim(U, i, j)

    def ang_U(i: int, j: int) -> float:
        c = float(np.clip(np.dot(U[i], U[j]), -1.0, 1.0))
        return float(np.arccos(c))

    return {
        "euclidean_X": (eucl_X, True),
        "sqeuclidean_X": (sqeucl_X, False),
        "cosine_X": (cos_X, False),
        "cosine_U": (cos_U, False),
        "angular_U": (ang_U, True),
    }


def violation_rate(d: Callable[[int, int], float]) -> float:
    viol = 0
    for t in range(trials):
        a, b, c = int(triples[t, 0]), int(triples[t, 1]), int(triples[t, 2])
        da, db, dc = d(a, b), d(b, c), d(a, c)
        if da + db < dc - tol or da + dc < db - tol or db + dc < da - tol:
            viol += 1
    return viol / trials


distances = make_distances()
print(f"n={n:,}  trials={trials:,}  tol={tol}  (same triples for all distances)\n")
for name, (fn, is_metric) in distances.items():
    rate = violation_rate(fn)
    flag = "metric" if is_metric else "non-metric"
    print(f"{name:16s}  violation_rate={rate:.6f}  ({flag})")
